# Лабораторная работа 4
**Тема.** 

**Задача 1.**
$$
P_3 | \space | C_{max}
$$ 

Алгоритм: longest processing time first (lpt) rule  (3.1.1)

In [1]:
jobs = {
    'J1': 7, 
    'J2': 8, 
    'J3': 4, 
    'J4': 9, 
    'J5': 12, 
    'J6': 5, 
    'J7': 3, 
    'J8': 9, 
    'J9': 5, 
    'J10': 12, 
    'J11': 7, 
    'J12': 5, 
    'J13': 8
}

In [2]:
sorted_jobs = sorted(jobs.items(), key=lambda x: x[1], reverse=True)

machines = [0, 0, 0]
schedule = {0: [], 1: [], 2: []}

for j_id, p in sorted_jobs:
    m_idx = machines.index(min(machines))
    
    start = machines[m_idx]
    end = start + p
    machines[m_idx] = end
    schedule[m_idx].append((j_id, start, end))

In [3]:
res_sched, res_cmax = schedule, max(machines)

for m, ops in res_sched.items():
    print(f"Станок {m+1}: {ops}")
print(f"C_max: {res_cmax}")

Станок 1: [('J5', 0, 12), ('J2', 12, 20), ('J11', 20, 27), ('J3', 27, 31)]
Станок 2: [('J10', 0, 12), ('J13', 12, 20), ('J6', 20, 25), ('J9', 25, 30), ('J7', 30, 33)]
Станок 3: [('J4', 0, 9), ('J8', 9, 18), ('J1', 18, 25), ('J12', 25, 30)]
C_max: 33


**Задача 2.**
$$
P_3 | prmp | C_{max}
$$ 

Алгоритм: direct graph technique (правило Макнотона) (3.3)

In [4]:
total_p = sum(jobs.values())
max_p = max(jobs.values())
c_max = max(max_p, total_p / m)

schedule = {i: [] for i in range(1, m + 1)}
current_m = 1
current_t = 0

for j_id, p in jobs.items():
    remaining_p = p
    while remaining_p > 0:
        capacity = c_max - current_t
        if remaining_p <= capacity:
            schedule[current_m].append((j_id, current_t, current_t + remaining_p))
            current_t += remaining_p
            remaining_p = 0
        else:
            schedule[current_m].append((j_id, current_t, c_max))
            remaining_p -= capacity
            current_m += 1
            current_t = 0

In [5]:
c_max_val, final_sched =  c_max, schedule

print(f"Оптимальный C_max: {c_max_val:.2f}")
for m_id, pieces in final_sched.items():
    print(f"Станок {m_id}: {pieces}")

Оптимальный C_max: 47.00
Станок 1: [('J1', 0, 7), ('J2', 7, 15), ('J3', 15, 19), ('J4', 19, 28), ('J5', 28, 40), ('J6', 40, 45), ('J7', 45, 47.0)]
Станок 2: [('J7', 0, 1.0), ('J8', 1.0, 10.0), ('J9', 10.0, 15.0), ('J10', 15.0, 27.0), ('J11', 27.0, 34.0), ('J12', 34.0, 39.0), ('J13', 39.0, 47.0)]


**Задача 3.**
$$
Q_4 | prmp | C_{max}
$$ 

Алгоритм: LRPT-FM 
Assign jobs with largest remaining process time (LRPT) to fastest machines (FM). Every time a fastest machine completes a job, the job on the second fastest machine is transferred to fastest machine, and , so on. This rule is called LRPT-FM rule. 
(3.7)

In [6]:
speeds = [3, 4, 3, 2]

In [7]:
p_rem = {j: float(p) for j, p in jobs.items()}
speeds = sorted(speeds, reverse=True)
n_m = len(speeds)

t = 0
dt = 0.01
history = []

while sum(p_rem.values()) > 0.001:
    active_jobs = sorted([j for j in p_rem if p_rem[j] > 0], 
                        key=lambda x: p_rem[x], reverse=True)

    for i in range(min(len(active_jobs), n_m)):
        job_id = active_jobs[i]
        s = speeds[i]
        
        consumption = s * dt
        if p_rem[job_id] < consumption:
            consumption = p_rem[job_id]
        
        p_rem[job_id] -= consumption
        history.append((job_id, i+1, t, t + dt))
        
    t += dt
    if t > 1000: break

In [8]:
final_cmax, _ = t, history
print(f"C_max для Q4 по правилу LRPT-FM: {final_cmax:.2f}")

C_max для Q4 по правилу LRPT-FM: 7.85


**Задача 4.**
$$
P_3 | | C_{max}
$$ 
 с помощью полиномиальной приближенной схемы с точностью $ε = 1/ 5 $

 Алгоритм: полиномиальной приближенной схемы (PTAS)

In [9]:
m = 3
epsilon = 1/5

In [10]:
total_load = sum(jobs.values())
max_job = max(jobs.values())
lb = max(total_load / m, max_job)

threshold = lb * epsilon

large_jobs = {k: v for k, v in jobs.items() if v > threshold}
small_jobs = {k: v for k, v in jobs.items() if v <= threshold}

machine_loads = [0] * m
schedule = [[] for _ in range(m)]

sorted_large = sorted(large_jobs.items(), key=lambda x: x[1], reverse=True)

for j_id, p in sorted_large:
    m_idx = machine_loads.index(min(machine_loads))
    machine_loads[m_idx] += p
    schedule[m_idx].append((j_id, p))

for j_id, p in small_jobs.items():
    m_idx = machine_loads.index(min(machine_loads))
    machine_loads[m_idx] += p
    schedule[m_idx].append((j_id, p))

In [11]:
res_schedule, loads, lower_bound = schedule, machine_loads, lb

print(f"Нижняя граница (LB): {lower_bound:.2f}")
print(f"Целевой максимум (1+eps)*LB: {lower_bound * (1 + epsilon):.2f}")
print("-" * 30)

for i, load in enumerate(loads):
    job_list = ", ".join([f"{j[0]}({j[1]})" for j in res_schedule[i]])
    print(f"M{i+1} [Итого {load}]: {job_list}")

print("-" * 30)
print(f"Полученный C_max: {max(loads)}")

Нижняя граница (LB): 31.33
Целевой максимум (1+eps)*LB: 37.60
------------------------------
M1 [Итого 32]: J5(12), J2(8), J11(7), J9(5)
M2 [Итого 29]: J10(12), J13(8), J3(4), J6(5)
M3 [Итого 33]: J4(9), J8(9), J1(7), J7(3), J12(5)
------------------------------
Полученный C_max: 33
